## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Import.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
from huggingface_hub import hf_hub_download

Configure root.

In [2]:
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")

if COLAB:
    root = Path("/content/mini-gLM")

    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])

else:
    root = Path.cwd().parent

sys.path.insert(0, str(root))

Try 512, then 1024 vocab size.

In [3]:
from src.tokenize import BPETokenizer

path = hf_hub_download(
    repo_id="eddykang06/hg38-pretraining",
    repo_type="dataset",
    filename="pretraining.csv.gz",
)

pretraining = pd.read_csv(path, compression = "gzip")

sequence_list = pretraining["sequence"].to_list()

tokenizer = BPETokenizer()
tokenized = tokenizer.train_tokenize(
    sequence_list = sequence_list,
    final_vocab_size = 512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


pretraining.csv.gz:   0%|          | 0.00/772M [00:00<?, ?B/s]

KeyboardInterrupt: 

Write to parquet file, also get the merge rules and token ID mappings.

In [ ]:
import pickle

pretraining.drop(columns = ["sequence"], inplace= True)
pretraining["tokenized_sequence"] = tokenized

pretraining.to_parquet("tokenized-512.parquet", engine='pyarrow', compression='gzip', index = False)
merge_rules = tokenizer.merge_rules
token_to_idx = tokenizer.token_to_idx

with open("merge-rules-512.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

with open("token-map-512.pkl", "wb") as f:
    pickle.dump(token_to_idx, f)

from google.colab import files

files.download("tokenized-512.parquet")
files.download("merge-rules-512.pkl")
files.download("token-map-512.pkl")

## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Implement ALiBi relative positional encoding.

In [ ]:
class ALiBi(nn.Module):
    def __init__(self, num_heads):
        super().__init__()

        self.num_heads = num_heads
    
    def forward(self, x):

        # Get shape
        B, L, _ = x.shape
        device = x.device

        # positions [L]
        positions = torch.arange(L, device = device)
        
        # dist [L, L]
        dist = -torch.abs(positions[:, None] - positions[None, :])
        
        # slopes [num_heads]
        init_slope = 2**(-8 / self.num_heads)
        slopes = torch.full((self.num_heads,), init_slope, device = device)
        slopes = torch.cumprod(slopes, dim = 0)
        
        # biases [num_heads, L, L]
        biases = dist.unsqueeze(0).expand(self.num_heads, -1, -1)
        biases = slopes[:, None, None] * biases

        # out [B, num_heads, L, L]
        out = biases.unsqueeze(0).expand(B, -1, -1, -1)

        return out

NameError: name 'nn' is not defined

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = self.d_model // self.num_heads

        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

        # Alibi positional encodings
        self.alibi = ALiBi(num_heads = self.num_heads)

        # Final FC
        self.o_map = nn.Linear(d_model, d_model)

    def forward(self, x, mask):

        B, L, D = x.shape

        q = self.q_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / (self.d_head ** 0.5)

        # Add alibi scores
        scores = scores + self.alibi(x)

        # Padding mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        a = torch.softmax(scores, dim = -1)
        
        out = a @ v
        out = out.transpose(1, 2).reshape(B, L, D)
        out = self.o_map(out)

        return out

# Note: make sure tha mask is [B, 1, 1, L]

Implement multi-head sparse attention block with attention mask.

In [ ]:
class MultiHeadSparseAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement Mixture of Experts block.

- https://huggingface.co/blog/moe
- https://machinelearningmastery.com/mixture-of-experts-architecture-in-transformer-models/

In [ ]:
# SwiGLU style expert
class SwiGLU(nn.Module):
    def __init__(self, input_dim, h_dim):
        super().__init__()

        self.input_dim = input_dim
        self.h_dim = h_dim

        self.gate_proj = nn.Linear(input_dim, h_dim)
        self.up_proj = nn.Linear(input_dim, h_dim)
        self.down_proj = nn.Linear(h_dim, input_dim)
        self.act = nn.SiLU()

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        swish = self.act(gate)
        out = self.down_proj(swish * up)

        return out
    

class MoELayer(nn.Module):
    def __init__(self, input_dim, h_dim, num_experts, top_k):
        super().__init__()
        
        self.input_dim = input_dim
        self.h_dim = h_dim,
        self.num_experts = num_experts
        self.top_k = top_k

        # Initialize the swiglu experts
        self.experts = nn.ModuleList([
            SwiGLU(input_dim, h_dim) for _ in range(num_experts)
        ])

        # Router for per-expert logits
        self.router = nn.Linear(input_dim, num_experts)

    def forward(self, x):

        B, L, D = x.shape

        # Reshape for expert processing [B * L, D]
        x_reshaped = x.reshape(-1, D)

        # Logits to [B * L, num_experts]
        router_logits = self.router(x_reshaped)

        # Get the top-k experts, then softmax to probabilty distribution over those k experts
        # Output [B * L, k]
        top_k_logits, top_k_idx = torch.topk(router_logits, self.top_k, dim = -1)
        top_k_probs = F.softmax(top_k_logits, dim = -1)

        # Initialize output tensor
        out = torch.zeros(
            B * L, D,
            device = x.device,
            dtype = x.dtype
        )

        # Process through selected experts
        unique_experts = torch.unique(top_k_idx)

        for i in unique_experts:
            expert_id = int(i)

            # Token mask [B*L] to decide which token of input should use this expert
            mask = (top_k_idx == expert_id)
            token_mask = mask.any(dim = 1)
            assert token_mask.any()

            # Select tokens, apply the expert, and add to output
            expert_input = x_reshaped[token_mask]
            expert_weight = top_k_probs[mask].unsqueeze(-1) # [N, 1]
            expert_output = self.experts[expert_id](expert_input) # [N, hidden_dim]

            out[token_mask] += expert_output * expert_weight

        # Reshape to original
        out = out.reshape(B, L, D)

        return out

Implement transformer encoder block with mixture of experts.

In [ ]:
class MoETransformer():
    def __init__(self, d_model, num_heads, h_dim, num_experts, top_k, p_drop):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k

        # Layers
        self.attention = MultiHeadAttention(
            d_model = self.d_model,
            num_heads = self.num_heads
        )
        self.dropout1 = nn.Dropout(p = p_drop)
        self.norm1 = nn.LayerNorm()
        self.moe = MoELayer(
            input_dim = self.d_model,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k
        )
        self.dropout2 = nn.Dropout(p = p_drop)
        self.norm2 = nn.LayerNorm()
    
    def forward(self, x):

        x = self.norm1(x + self.dropout1(self.attention(x))) 
        out = self.norm2(x + self.dropout2(self.moe(x)))

        return out

Implement models and compute mask for padding tokens???? Padding as "PAD"

In [ ]:
class DenseGLM(nn.Module):
    def __init__(self, vocab_size, num_blocks, d_model, num_heads, h_dim, num_experts, top_k, p_drop):
        
        self.vocab_size = vocab_size
        self.num_blocks = num_blocks
        self.d_model = d_model
        self.num_heads = num_heads
        self.h_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.p_drop = p_drop

        self.embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.d_model
        )

        self.moe_transformer = MoETransformer(
            d_model = self.d_model,
            num_heads = self.num_heads,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k,
            p_drop = self.p_drop
        )

        self.model = nn.ModuleList([
            self.moe_transformer for _ in range(num_blocks)
        ])

        # Final mapping to vocab size
        self.final = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):

        x = self.model(x)
        logits = self.final(x)

        return logits

## Training

Implement masked language modeling training objective.
- Embedding module
- Relative positional embeddings
- Batching [B, L, D]
    - Token right padding to max length in batch
- Attention mask to ignore padding tokens
- Max sequence length cutoff
- Masked token prediction objective
Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

Train tokenizer.

In [ ]:
from src.tokenize import BPETokenizer
from torch.utils.data import DataLoader, Dataset


class hg38Data(Dataset):
    """Custom dataset to load hg38 pre-training data from HuggingFace"""
    def __init__(self, tokenized_path):
        
        # Get sequences from HF path to tokenized dataset
        self.tokenized_path = tokenized_path

        # Convert the parquet file to a list of sequences lol
        self.tokenized_list 

    def __len__(self):
        len(self.tokenized_list)
    
    def __getitem__(self, idx):
        sequence = self.tokenized_list[idx]
        sequence = torch.tensor(sequence)

        return sequence


# Implement a data loader with dynamic batching and padding to max batch sequence lnegth
class DynamicDataLoader():
    """Custom DataLoader to implement Dynamic batching """

Implementing dynamic batching (batch similar sequences together)

- collate_fn
Dynamically pad each batch only to the longest sequence in that batch. PyTorch DataLoader directly supports custom collate_fn.
- pad_sequence
Pads a list of variable-length tensors into one batch tensor. The padded length becomes the longest sequence in that batch.
- custom batch_sampler
Groups examples by length or by total token count. DataLoader supports batch_sampler, which lets you control exactly which indices go into each batch.

In [ ]:
from torch.utils.data import DataLoader, Dataset


dataset = tokenized
train_loader = DataLoader    

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import random_split


# Set seed
torch.manual_seed(111)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)